# Data overview — what exists, at which layer, right now

One page that answers "what data do I actually have?". Every number below is
counted from disk when the cell runs. Nothing is cached, simulated or filled in.

The layers, in medallion order:

| Layer | What it holds | Rebuildable |
| --- | --- | --- |
| **bronze** | original bytes plus an envelope pinning URL, time and SHA-256 | no — refetching gives new bytes |
| **silver** | parsed records, one provision or passage per row | yes, from bronze |
| **gold** | Parquet tables and the DuckDB projection | yes, from silver and run evidence |
| **runs** | raw API responses and the JSONL records of an experiment | **no — irreplaceable** |

Run evidence is not derived from bronze and cannot be rebuilt: a second model
call is new evidence, not a copy of the first. Back up `data/local/runs/`.

In [1]:
import json
import sys
from pathlib import Path

import yaml

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from simulacria.domains import get, names
from simulacria.generation.plan import load_config, plan
from simulacria.measurement.corpus import load_corpus

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)


def megabytes(paths) -> float:
    return round(sum(p.stat().st_size for p in paths) / 1e6, 2)


print(f"Project root: {ROOT}")

Project root: C:\Users\46762\VSCODE\allegoria


## 1. Bronze — original bytes, one row per domain

A domain adapter owns its bronze layout and verifies it on every load: the
bytes are re-hashed and compared with the envelope before any text is used.

In [2]:
rows = []
for name in names():
    domain = get(name)
    if not domain.source_dir:
        rows.append({"domain": name, "language": domain.language, "documents": "inline corpus"})
        continue
    source = (
        sorted((ROOT / domain.source_dir).glob("*")) if (ROOT / domain.source_dir).is_dir() else []
    )
    bronze = (
        sorted((ROOT / domain.bronze_dir).glob("*.json"))
        if (ROOT / domain.bronze_dir).is_dir()
        else []
    )
    rows.append(
        {
            "domain": name,
            "language": domain.language,
            "documents": len(bronze),
            "source_MB": megabytes(source),
            "bronze_MB": megabytes(bronze),
            "source_dir": domain.source_dir,
            "id_format": domain.id_format,
        }
    )
display(pd.DataFrame(rows))

,domain,language,documents,source_MB,bronze_MB,source_dir,id_format
0,inline,any,inline corpus,NaN,NaN,NaN,NaN
1,rfc,en,2,0.08,0.09,data/source/rfc,"rfc-<number>:<section>, e.g. rfc-2119:4"
2,sfs,sv,50,2.70,4.83,data/source/sfs,"<document id>:<provision suffix>, e.g. sfs-1982-673:P13"


## 2. Silver — parsed records

The SFS pool is parsed into `provisions.jsonl`; the v1 count is a contract that
the build refuses to violate. Other domains are read per passage on demand.

In [3]:
silver = []
for label, relative in (
    ("v1", "data/local/provisions.jsonl"),
    ("v2", "data/local/provisions_v2.jsonl"),
):
    path = ROOT / relative
    if path.is_file():
        lines = path.read_text(encoding="utf-8").splitlines()
        documents = {json.loads(line)["document_id"] for line in lines if line}
        silver.append(
            {
                "pool": label,
                "provisions": len(lines),
                "documents": len(documents),
                "MB": megabytes([path]),
                "path": relative,
            }
        )
    else:
        silver.append(
            {
                "pool": label,
                "provisions": 0,
                "documents": 0,
                "MB": 0.0,
                "path": f"{relative} (not built)",
            }
        )
display(pd.DataFrame(silver))

,pool,provisions,documents,MB,path
0,v1,1952,50,2.64,data/local/provisions.jsonl
1,v2,18836,466,29.21,data/local/provisions_v2.jsonl


## 3. The annotated corpora — what the experiment actually reads

These are the passages with hand-drafted slots. Draft means draft: no human has
reviewed them, and every number computed from them carries that caveat.

In [4]:
corpora = []
for path in sorted((ROOT / "corpus").glob("*.yaml")):
    spec = yaml.safe_load(path.read_text(encoding="utf-8"))
    if spec.get("schema_version") != 2 or "domain" not in spec:
        corpora.append({"corpus": path.name, "status": "not an annotated corpus (axis input)"})
        continue
    passages = load_corpus(path, ROOT)
    corpora.append(
        {
            "corpus": path.name,
            "domain": spec["domain"],
            "passages": len(passages),
            "slots": sum(len(p["slots"]) for p in passages),
            "chars_median": int(pd.Series([len(p["text"]) for p in passages]).median()),
            "groups": ", ".join(sorted({p["group"] for p in passages})),
            "status": passages[0]["annotation_status"],
        }
    )
display(pd.DataFrame(corpora))

,corpus,domain,passages,slots,chars_median,groups,status
0,law_probe_v1.yaml,sfs,3.0,15.0,321.0,statutory,assistant_draft_requires_human_review
1,philosophy_v1.yaml,inline,30.0,90.0,318.0,"categorical, conditional, virtue",assistant_draft_requires_human_review
2,rfc_probe_v1.yaml,rfc,3.0,12.0,472.0,technical_norm,assistant_draft_requires_human_review
3,statutory_axis_v1.yaml,NaN,NaN,NaN,NaN,NaN,not an annotated corpus (axis input)


## 4. Configured experiments — what a run would cost before you pay for it

In [5]:
experiments = []
for path in sorted((ROOT / "configs").glob("*.yaml")):
    config = load_config(ROOT, path.relative_to(ROOT).as_posix())
    sources, chains = plan(ROOT, config)
    transformations = len(chains) * config["generations"]
    experiments.append(
        {
            "experiment": config["experiment"],
            "language": config.get("language"),
            "passages": len(sources),
            "chains": len(chains),
            "depth": config["generations"],
            "transformations": transformations,
            "readings": transformations + len(sources),
            "config": path.name,
        }
    )
display(pd.DataFrame(experiments))

,experiment,language,passages,chains,depth,transformations,readings,config
0,allegoria-sv,sv,33,84,10,840,873,allegoria-sv.yaml
1,pilot-sv,sv,3,24,1,24,27,pilot-sv.yaml
2,rfc-en,en,3,24,10,240,243,rfc-en.yaml


## 5. Gold — the Parquet tables and the DuckDB projection

In [6]:
tables = ROOT / "data/local/tables"
if tables.is_dir():
    display(
        pd.DataFrame(
            [
                {"table": p.stem, "MB": megabytes([p]), "path": p.relative_to(ROOT).as_posix()}
                for p in sorted(tables.glob("*.parquet"))
            ]
        )
    )
else:
    print("data/local/tables is missing - run python scripts/pipeline/build_tables.py")

database = ROOT / "data/local/allegoria.duckdb"
if database.is_file():
    import duckdb

    with duckdb.connect(str(database), read_only=True) as connection:
        listed = connection.execute(
            "select table_schema, table_name from information_schema.tables "
            "where table_schema in ('research', 'selection') order by 1, 2"
        ).fetchall()
        display(
            pd.DataFrame(
                [
                    {
                        "schema": schema,
                        "table": table,
                        "rows": connection.execute(
                            f'select count(*) from "{schema}"."{table}"'
                        ).fetchone()[0],
                    }
                    for schema, table in listed
                ]
            )
        )
else:
    print(f"{database.name} is missing - run python scripts/pipeline/build_database.py")

,table,MB,path
0,candidates,0.01,data/local/tables/candidates.parquet
1,marker_hits,0.06,data/local/tables/marker_hits.parquet
2,markers,0.00,data/local/tables/markers.parquet
3,provisions,0.40,data/local/tables/provisions.parquet


,schema,table,rows
0,research,call_events,0
1,research,readings,0
2,research,runs,0
3,research,slot_comparisons,0
4,research,slot_observations,0
5,research,source_slots,0
6,research,texts,0
7,selection,candidates,461
8,selection,marker_hits,8983
9,selection,markers,28


## 6. Runs — the evidence that cannot be rebuilt

Status is read from each manifest. `partial` means exactly that, and a run
recorded with models this code no longer supports is listed as unreadable
rather than quietly skipped.

In [7]:
from simulacria.generation.provider import supported

runs = []
for manifest_path in sorted((ROOT / "data/local/runs").glob("*/manifest.json")):
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    models = (manifest.get("transformer_model", ""), manifest.get("extractor_model", ""))
    directory = manifest_path.parent
    runs.append(
        {
            "run": directory.name[:28],
            "experiment": manifest.get("experiment", "(pre-config run)"),
            "status": manifest.get("status"),
            "generations": manifest.get(
                "completed_transformations", len(list(directory.glob("generations.jsonl")))
            ),
            "readings": manifest.get("completed_readings"),
            "raw_files": len(list((directory / "raw").glob("*.json"))),
            "MB": megabytes(list(directory.rglob("*"))),
            "usd": manifest.get("usage_cost_estimate_usd"),
            "readable_now": all(supported(m) for m in models),
            "models": " / ".join(models),
        }
    )
display(pd.DataFrame(runs) if runs else "No runs yet. Nothing has been generated.")

,run,experiment,status,generations,readings,raw_files,MB,usd,readable_now,models
0,recursive-81c11f28ed4541158f,(pre-config run),partial,35,45,192,1.08,0.068162,False,gpt-5.4-nano-2026-03-17 / gpt-5.4-mini-2026-03-17


## 7. What to back up

Everything below is either irreplaceable evidence or cheap to rebuild. The
distinction is the only one that matters when clearing space.

In [8]:
display(
    pd.DataFrame(
        [
            {
                "path": "data/source/",
                "kind": "original bytes",
                "rebuildable": "no (refetch differs)",
                "backup": "yes",
            },
            {"path": "data/bronze/", "kind": "envelopes", "rebuildable": "no", "backup": "yes"},
            {
                "path": "data/local/runs/",
                "kind": "API receipts and run records",
                "rebuildable": "no",
                "backup": "yes, first",
            },
            {
                "path": "corpus/, prompts/, configs/",
                "kind": "experiment specification",
                "rebuildable": "no",
                "backup": "in git",
            },
            {
                "path": "data/local/provisions*.jsonl",
                "kind": "silver",
                "rebuildable": "yes",
                "backup": "no",
            },
            {
                "path": "data/local/tables*/",
                "kind": "gold Parquet",
                "rebuildable": "yes",
                "backup": "no",
            },
            {
                "path": "data/local/allegoria.duckdb",
                "kind": "gold projection",
                "rebuildable": "yes",
                "backup": "no",
            },
        ]
    )
)

,path,kind,rebuildable,backup
0,data/source/,original bytes,no (refetch differs),yes
1,data/bronze/,envelopes,no,yes
2,data/local/runs/,API receipts and run records,no,"yes, first"
3,"corpus/, prompts/, configs/",experiment specification,no,in git
4,data/local/provisions*.jsonl,silver,yes,no
5,data/local/tables*/,gold Parquet,yes,no
6,data/local/allegoria.duckdb,gold projection,yes,no
